# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Let's inspect the available record sets, fields, and columns in this Croissant dataset. All entities (record sets, fields, columns) will be referenced by their `@id` as per the FAIR principles.

In [ ]:
# Get all RecordSets in the dataset
record_sets = metadata.record_sets
print(f"Number of record sets in the dataset: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    print("  Fields and Columns:")
    # For each field, print its @id and the underlying column @id
    for field in rs.fields:
        print(f"    Field name: {field.name}")
        print(f"      Field @id: {field.id}")
        # Each field may have one or more columns
        columns = getattr(field, 'columns', [])
        if len(columns) > 0:
            for col in columns:
                print(f"      Column @id: {col.id} (name: {col.name})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract data from ALL record sets and store them in a dictionary of DataFrames. All references use the `@id`s as required.

In [ ]:
# Build a list of record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Available record_set @id values:", record_set_ids)

# Load records for all record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet '@id': {record_set_id} with {len(df)} rows and {len(df.columns)} columns.")

# Display columns and a sample for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nRecordSet @id: {first_rs} columns: \n{dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering records, normalizing a numeric field, and grouping. 

Below is an example using a selected numeric field and group field. **Replace `<numeric_field_id>` and `<group_field_id>` with actual field/column names or their corresponding `@id`s based on your record set**.

_The following block is tailored for the first record set for demonstration; to work with other record sets or fields, adjust the field variables as needed._

In [ ]:
# Pick the first record set for EDA
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]
print(f"RecordSet @id for EDA: {record_set_id}")

# List available columns
print("Columns:", df.columns.tolist())

# --- Set these variables based on actual fields observed in data overview ---
# As an example, use a common numeric field such as 'MolecularWeight' or similar if present.
# Replace these with actual @id or column names as needed.
# For demonstration, try to find a plausible numeric and group field from columns

possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or df[col].apply(lambda v: str(v).replace('.','',1).isdigit() if pd.notnull(v) else False).all()]
print("Possible numeric fields:", possible_numeric_fields)
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    numeric_field = df.columns[0]  # fallback, for demonstration

print(f"Selected numeric field for filtering and normalization: {numeric_field}")

# Choose a group field if present
possible_group_fields = [col for col in df.columns if df[col].nunique() < len(df)//2 and df[col].dtype == object]
group_field = possible_group_fields[0] if possible_group_fields else None
print(f"Selected group field for grouping: {group_field}")

# Ensure numeric_field is numeric for calculations
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# 1. Filtering: Keep rows where numeric_field > mean
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.4f} (mean): {len(filtered_df)} rows")
display(filtered_df.head())

# 2. Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First 5 normalized {numeric_field} values:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# 3. Optional: Group by group_field and show aggregated means
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Mean {numeric_field} per {group_field} (for filtered rows):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we demonstrate histograms and scatter plots using the selected numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
plt.title(f"Distribution of Normalized {numeric_field}")
plt.xlabel(f"{numeric_field}_normalized")
plt.ylabel("Frequency")
plt.show()

# If group_field available, visualise group means
if group_field:
    plt.figure(figsize=(10, 4))
    order = filtered_df[group_field].value_counts().index
    sns.barplot(
        data=filtered_df,
        x=group_field, y=numeric_field,
        estimator='mean', errorbar='ci', order=order
    )
    plt.title(f"Mean {numeric_field} by {group_field} (filtered)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# If two numeric fields, do scatter
if len(possible_numeric_fields) >= 2:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(
        data=filtered_df,
        x=possible_numeric_fields[0],
        y=possible_numeric_fields[1],
        hue=group_field if group_field else None
    )
    plt.title(f"Scatter: {possible_numeric_fields[0]} vs {possible_numeric_fields[1]}")
    plt.xlabel(possible_numeric_fields[0])
    plt.ylabel(possible_numeric_fields[1])
    plt.legend()
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook we:

- Loaded the Croissant-structured dataset using `mlcroissant`
- Explored available record sets, fields, and columns (referenced by their `@id`s)
- Extracted records to pandas DataFrames
- Performed exploratory analysis: filtering, normalization, grouping
- Visualized key distributions and relationships

To adapt this notebook to your specific downstream analysis, select the relevant record set and fields by their `@id`, and perform domain-specific feature engineering or modeling as needed.